# Datasets

> Download and store datasets

In [ ]:
#| default_exp datasets

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import os
from pathlib import Path

from pooch import create as pooch_create, Untar, Unzip, Decompress
import quilt3
import pandas as pd
import numpy as np
import medmnist

from PIL import Image
import tifffile as tiff
from tqdm import tqdm


### MedMNIST Datasets

In [ ]:
#| export
import os
from tqdm import tqdm
from PIL import Image
import medmnist
import tifffile as tiff

def download_medmnist(dataset: str, # The name of the MedMNIST dataset (e.g., 'pathmnist', 'bloodmnist', etc.).
                      output_dir: str = '.', # The path to the directory where the datasets will be saved.
                      download_only: bool = False, # If True, only download the dataset into the output directory without processing.
                      save_images: bool = True, # If True, save the images into the output directory as .png (2D datasets) or multipage .tiff (3D datasets) files.
                      ):
    """
    Downloads the specified MedMNIST dataset and saves the training, validation, and test datasets 
    into the specified output directory. Images are saved as .png for 2D data and multi-page .tiff for 3D data,
    organized into folders named after their labels.

    Args:
    - dataset: The MedMNIST dataset name (e.g., 'pathmnist', 'bloodmnist', etc.).
    - output_dir: Path where the images will be saved.
    - download_only: If True, only downloads the dataset, no processing or saving.
    - save_images: If True, save the images in the specified output directory.

    Returns:
    - None, saves images in the specified output directory if save_images is True.
    """

    # Check if the dataset is available in the MedMNIST information dictionary
    if dataset not in medmnist.INFO:
        raise ValueError(f"The dataset '{dataset}' is not available. Please select from the available datasets.")

    # Retrieve dataset information
    info = medmnist.INFO[dataset]

    # Get the appropriate dataset class from MedMNIST
    dataset_class = getattr(medmnist, info['python_class'])

    # Create the output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Download the datasets
    train_dataset = dataset_class(split='train', download=True, root=output_dir)
    val_dataset = dataset_class(split='val', download=True, root=output_dir)
    test_dataset = dataset_class(split='test', download=True, root=output_dir)

    # Save the images into directories by their label
    def save_images(dataset, split):
        """Helper function to save images and labels into directories."""
        split_dir = os.path.join(output_dir, split)
        if not os.path.exists(split_dir):
            os.makedirs(split_dir)

        for i in tqdm(range(len(dataset))):
            img, label = dataset[i]
            label_dir = os.path.join(split_dir, str(label).replace("[", "").replace("]", ""))  # Remove parentheses
            if not os.path.exists(label_dir):
                os.makedirs(label_dir)

            # Save 2D images as .png
            if info['n_channels'] == 1:  # Check if it's 2D (single-channel)
                img_path = os.path.join(label_dir, f'{split}_{i}.png')
                img = Image.fromarray(img.squeeze(), mode='L')  # 'L' mode for grayscale
                img.save(img_path)
            elif info['n_channels'] == 3:  # Check if it's RGB
                img_path = os.path.join(label_dir, f'{split}_{i}.png')
                img.save(img_path)
            # Save 3D images as multi-page .tiff
            else:
                img_path = os.path.join(label_dir, f'{split}_{i}.tiff')
                tiff.imwrite(img_path, img)

    # Save training, validation, and test data if save_images is True
    if save_images:
        print(f"Saving training images to {output_dir}...")
        save_images(train_dataset, 'train')

        print(f"Saving validation images to {output_dir}...")
        save_images(val_dataset, 'val')

        print(f"Saving test images to {output_dir}...")
        save_images(test_dataset, 'test')
        
        # Clean up: remove .npz files if present
        for file in os.listdir(output_dir):
            if file.endswith('.npz'):
                os.remove(os.path.join(output_dir, file))
                print(f"Removed {file}")

    # If download_only is True, skip returning the dataset objects and just download the files
    if download_only:
        print(f"Datasets downloaded to {output_dir}")
        print(f"Dataset info for '{dataset}': {info}")
        return info

    # Return the datasets if download_only is False and save_images is False
    return train_dataset, val_dataset, test_dataset if not save_images else None


In [ ]:
#| export
def medmnist2df(train_dataset,     # MedMNIST training dataset with images and labels
                val_dataset=None,  # (Optional) MedMNIST validation dataset with images and labels
                test_dataset=None, # (Optional) MedMNIST test dataset with images and labels
                mode='RGB'         # Mode for PIL Image conversion, e.g., 'RGB', 'L'
               ) -> (pd.DataFrame, pd.DataFrame, pd.DataFrame): # (df_train, df_val, df_test): DataFrames with columns 'image' and 'label'
    """
    Convert MedMNIST datasets to DataFrames, with images as PIL Image objects and labels as DataFrame columns.
    
    Missing datasets (if None) are represented by None in the return tuple.
    """
    
    # Helper function to convert dataset to DataFrame
    def dataset_to_df(dataset, mode):
        images, labels = dataset.imgs, dataset.labels
        return pd.DataFrame({
            'image': [Image.fromarray(img, mode) for img in images], 
            'label': labels.squeeze()
        })
    
    # Convert each dataset to a DataFrame if it exists
    df_train = dataset_to_df(train_dataset, mode) if train_dataset is not None else None
    df_val = dataset_to_df(val_dataset, mode) if val_dataset is not None else None
    df_test = dataset_to_df(test_dataset, mode) if test_dataset is not None else None
    
    return df_train, df_val, df_test


### Download data via Pooch

In [ ]:
#| export
def download_dataset(base_url, expected_checksums, file_names, output_dir, processor=None):
    """
    Download a dataset using Pooch and save it to the specified output directory.
    
    Parameters:
        base_url (str): The base URL from which the files will be downloaded.
        expected_checksums (dict): A dictionary mapping file names to their expected checksums.
        file_names (dict): A dictionary mapping task identifiers to file names.
        output_dir (str): The directory where the downloaded files will be saved.
        processor (callable, optional): A function to process the downloaded data. Defaults to None.
    """
    # Create a Pooch object with the base URL, output directory, and expected checksums
    pooch_instance = pooch_create(
        path=output_dir,
        base_url=base_url,
        registry=expected_checksums,
    )
    
    # Create the output directory if it does not exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Download each file if it is not already present in the output directory
    for _, file_name in file_names.items():
        pooch_instance.fetch(file_name, progressbar=True, processor=processor)
    
    print("The dataset has been successfully downloaded and saved to:", output_dir)


In [ ]:
#| hide

# # Specify the directory where you want to save the downloaded files
output_directory = "./_test_folder"

# Define the base URL for the MSD dataset
base_url = 'https://s3.ap-northeast-1.wasabisys.com/gigadb-datasets/live/pub/10.5524/100001_101000/100888/'

# Define the expected checksums for the files in the dataset
expected_checksums = {
'Samples/actin-20x-noise1-highsnr-sample.png': 'md5:7995383f95473a4e74a3b49ed2d6a846'
}

# Define the names of the files to be downloaded
file_names = {
'1': 'Samples/actin-20x-noise1-highsnr-sample.png'
}

# Download the dataset
download_dataset(base_url, expected_checksums, file_names, output_directory)










100%|█████████████████████████████████████| 2.38M/2.38M [00:00<00:00, 1.75GB/s]

The dataset has been successfully downloaded and saved to: ./_test_folder


In [ ]:
#| export
def download_dataset_from_csv(csv_file, base_url, output_dir, processor=None, rows=None, prepend_mdf5=True):
    """
    Download a dataset using Pooch and save it to the specified output directory, reading file names and checksums from a CSV file.

    Parameters:
        csv_file (str): Path to the CSV file containing file names and checksums.
        base_url (str): The base URL from which the files will be downloaded.
        output_dir (str): The directory where the downloaded files will be saved.
        processor (callable, optional): A function to process the downloaded data. Defaults to None.
        rows (list of int, optional): Specific row indices to download. If None, download all rows. Defaults to None.
    """
    # Read the CSV file
    df = pd.read_csv(csv_file)

    # If specific rows are provided, filter the DataFrame
    if rows is not None:
        df = df.iloc[rows]
        
    # Create a dictionary for expected checksums
    if prepend_mdf5:
        expected_checksums = pd.Series(df['MD5 Checksum'].apply(lambda x: f"md5:{x}").values, index=df['Filename']).to_dict()
    else:
        expected_checksums = pd.Series(df['MD5 Checksum'].values, index=df['Filename']).to_dict()

    # Create a Pooch object with the base URL, output directory, and expected checksums
    pooch_instance = pooch_create(
        path=output_dir,
        base_url=base_url,
        registry=expected_checksums,
    )
    
    # Create the output directory if it does not exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Download each file if it is not already present in the output directory
    for file_name in df['Filename']:
        pooch_instance.fetch(file_name, progressbar=True, processor=processor)
    
    print("The dataset has been successfully downloaded and saved to:", output_dir)



In [ ]:
#| hide

# Specify the directory where you want to save the downloaded files
output_directory = "./_test_folder"
# Define the base URL for the MSD dataset
base_url = 'https://s3.ap-northeast-1.wasabisys.com/gigadb-datasets/live/pub/10.5524/100001_101000/100888/'

download_dataset_from_csv('./data_examples/FMD_dataset_info.csv', base_url, output_directory, rows=[6])


The dataset has been successfully downloaded and saved to: ./_test_folder


### Download data via Quilt/T4

Allen Institute Cell Science (AICS)

In [ ]:
#| export

def aics_pipeline(n_images_to_download=40, image_save_dir=None): 
    # Set default save directory if not provided
    if image_save_dir is None:
        image_save_dir = os.getcwd()

    # Ensure the save directory exists; create it if not
    os.makedirs(image_save_dir, exist_ok=True)

    # Load the package
    package = quilt3.Package.browse("aics/pipeline_integrated_cell", registry="s3://allencell")
    
    # Load metadata
    data_manifest = package["metadata.csv"]()

    # Keep only unique FOVs
    unique_fov_indices = np.unique(data_manifest['FOVId'], return_index=True)[1]
    data_manifest = data_manifest.iloc[unique_fov_indices]

    # Select first n_images_to_download
    data_manifest = data_manifest.iloc[:n_images_to_download]

    # Get source and target paths
    image_source_paths = data_manifest["SourceReadPath"]
    image_target_paths = [os.path.join(image_save_dir, os.path.basename(image_source_path)) 
                          for image_source_path in image_source_paths]

    # Download images
    downloaded_image_paths = []
    for image_source_path, image_target_path in zip(image_source_paths, image_target_paths):
        if os.path.exists(image_target_path):
            downloaded_image_paths.append(image_target_path)
            continue  # Skip if already downloaded
        
        try:
            package[image_source_path].fetch(image_target_path)
            downloaded_image_paths.append(image_target_path)
        except (OSError, FileNotFoundError) as e:
            print(f"Failed to fetch {image_source_path}: {e}")
                
    return downloaded_image_paths, data_manifest


In [ ]:
image_target_paths, data_manifest = aics_pipeline(1, "../_data/aics")


Loading manifest: 100%|██████████| 77165/77165 [00:01<00:00, 44.8k/s]


In [ ]:
print(image_target_paths)
data_manifest.to_csv('../_data/aics/aics_dataset.csv')

['../_data/aics/6677e50c_3500001004_100X_20170623_5-Scene-1-P24-E06.ome.tiff']


### Dataset Manifest

Make a manifest of all of the files in csv form

In [ ]:
#| export
def manifest2csv(signal, target, paths=None, train_fraction=0.8, data_save_path='./', train='train.csv', test='test.csv', identifier=None):
    
    if paths is None:
        df = pd.DataFrame(columns=["path_signal", "path_target"])
        df["path_signal"] = signal
        df["path_target"] = target 
        length_dataset = len(signal)
    elif identifier is None:
        df = pd.DataFrame(columns=["path_tiff", "channel_signal", "channel_target"])
        df["path_tiff"] = paths
        df["channel_signal"] = signal
        df["channel_target"] = target 
        length_dataset = len(paths)
    else:
        df = pd.DataFrame(columns=["signal (path+identifier)", "target (path+identifier)"])
        df["signal (path+identifier)"] = paths
        df["signal (path+identifier)"] = df["signal (path+identifier)"] + identifier + [*signal.astype(str)]
        df["target (path+identifier)"] = paths
        df["target (path+identifier)"] = df["target (path+identifier)"] + identifier + [*target.astype(str)]       
        length_dataset = len(paths)
        

    n_train_images = int(length_dataset * train_fraction)
    df_train = df[:n_train_images]
    df_test = df[n_train_images:]

    df_test.to_csv(data_save_path+test, index=False)
    df_train.to_csv(data_save_path+train, index=False)

In [ ]:

manifest2csv(data_manifest["ChannelNumberBrightfield"],data_manifest["ChannelNumber405"], image_target_paths, data_save_path='./data_examples/')


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()